In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder
import optuna, warnings, subprocess, os, gc
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ── Environment ───────────────────────────────────────────────────────────────
IS_KAGGLE = os.path.exists("/kaggle/input")

def has_gpu():
    try: return subprocess.run(["nvidia-smi"], capture_output=True, timeout=5).returncode == 0
    except: return False

USE_GPU  = IS_KAGGLE or has_gpu()
DEVICE   = "cuda" if USE_GPU else "cpu"
DATA_DIR = "/kaggle/input/playground-series-s6e3/" if IS_KAGGLE else "../data/"
SUB_DIR  = "/kaggle/working/"                       if IS_KAGGLE else "../"
# Upload submission_xgb_scaledpos_blend.csv as dataset "churn-pseudo-inputs"
CSV_DIR  = "/kaggle/input/churn-pseudo-inputs/"     if IS_KAGGLE else "../"

print(f"Environment : {'Kaggle' if IS_KAGGLE else 'Local'}")
print(f"GPU         : {'Enabled ✓' if USE_GPU else 'CPU only'}")
print(f"XGBoost     : {xgb.__version__}")

TARGET   = "Churn"
SEEDS    = [42, 2024, 777, 1337, 999]
N_SPLITS = 10
TOTAL_MODELS = len(SEEDS) * N_SPLITS

# ── Precomputed best params from nb10 (skip tuning — save GPU time) ──────────
BEST_PARAMS = {
    "objective":        "binary:logistic",
    "eval_metric":      "auc",
    "tree_method":      "hist",
    "device":           DEVICE,
    "grow_policy":      "lossguide",
    "learning_rate":    0.02,
    "max_depth":        6,
    "max_leaves":       50,
    "min_child_weight": 5,
    "subsample":        0.8,
    "colsample_bytree": 0.8,
    "colsample_bylevel":0.8,
    "reg_alpha":        0.1,
    "reg_lambda":       1.0,
    "gamma":            0.05,
    "verbosity":        0,
    "nthread":          -1,
}
BEST_ROUNDS = 700


In [ ]:
train = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test  = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))

print(f"Train : {train.shape}  |  Test : {test.shape}")

train[TARGET] = (train[TARGET] == "Yes").astype(int)
churn_rate    = train[TARGET].mean()
scale_pos_w   = round((1 - churn_rate) / churn_rate, 4)
print(f"Churn rate : {churn_rate:.4f}  |  scale_pos_weight : {scale_pos_w}")
BEST_PARAMS["scale_pos_weight"] = scale_pos_w

sub = test[["id"]].copy()

# Joint preprocessing
combined  = pd.concat([train.drop(TARGET, axis=1), test], axis=0).reset_index(drop=True)
train_idx = range(len(train))
test_idx  = range(len(train), len(train) + len(test))

num_cols = combined.select_dtypes(include=np.number).columns.tolist()
cat_cols = [c for c in combined.select_dtypes(include="object").columns if c != "id"]

for c in num_cols: combined[c].fillna(combined[c].median(), inplace=True)
for c in cat_cols: combined[c].fillna("Missing", inplace=True)

# Label encode
le = LabelEncoder()
for c in cat_cols:
    combined[c] = le.fit_transform(combined[c].astype(str))

# Existing FE (proven 8 features from nb10)
num_base = ["tenure", "MonthlyCharges", "TotalCharges"]
combined["num_sum"]  = combined[num_base].sum(axis=1)
combined["num_mean"] = combined[num_base].mean(axis=1)
combined["num_std"]  = combined[num_base].std(axis=1)
combined["num_max"]  = combined[num_base].max(axis=1)
combined["num_min"]  = combined[num_base].min(axis=1)
combined["Average_Monthly_Actual"] = combined["TotalCharges"] / (combined["tenure"] + 1e-5)
combined["Monthly_diff"]           = combined["MonthlyCharges"] - combined["Average_Monthly_Actual"]
combined["Monthly_ratio"]          = combined["MonthlyCharges"] / (combined["Average_Monthly_Actual"] + 1e-5)

FEATURES = [c for c in combined.columns if c not in ["id", TARGET]]

train_df = combined.iloc[train_idx].copy()
test_df  = combined.iloc[test_idx].copy()

X      = train_df[FEATURES]
y      = train[TARGET]
X_test = test_df[FEATURES]

print(f"Features : {len(FEATURES)}  |  X : {X.shape}  |  X_test : {X_test.shape}")


In [ ]:
# Load nb10's best predictions (our current best: 0.91445)
nb10_preds = pd.read_csv(os.path.join(CSV_DIR, "submission_xgb_scaledpos_blend.csv"))
print(f"nb10 test preds loaded: {nb10_preds.shape}")
print(f"Pred range : [{nb10_preds[TARGET].min():.4f}, {nb10_preds[TARGET].max():.4f}]")
print(f"Pred mean  : {nb10_preds[TARGET].mean():.4f}  std: {nb10_preds[TARGET].std():.4f}")

# Merge predictions into test_df (align by id)
test_df_pl = test_df.copy()
test_df_pl["pred"] = nb10_preds[TARGET].values

print("\n── Pseudo-label counts at different thresholds ──")
print(f"  {'Threshold':15s} {'Churn=1':>10s} {'Churn=0':>10s} {'Total':>10s} {'% of test':>10s}")
print("  " + "-"*55)
for lo, hi in [(0.05,0.95),(0.08,0.92),(0.10,0.90),(0.15,0.85),(0.20,0.80)]:
    pos = (test_df_pl["pred"] > hi).sum()
    neg = (test_df_pl["pred"] < lo).sum()
    tot = pos + neg
    print(f"  {lo:.2f} / {hi:.2f}         {pos:>10,} {neg:>10,} {tot:>10,} {tot/len(test_df_pl)*100:>9.1f}%")


In [ ]:
# Two configs — conservative (higher quality) vs moderate (more data)
CONFIGS = [
    {"name": "pl_90_w05",  "thresh_hi": 0.90, "thresh_lo": 0.10, "pseudo_weight": 0.5},
    {"name": "pl_95_w05",  "thresh_hi": 0.95, "thresh_lo": 0.05, "pseudo_weight": 0.5},
]

def make_pseudo_dataset(cfg):
    """Build augmented train set: real train + high-confidence pseudo-labeled test."""
    hi, lo, pw = cfg["thresh_hi"], cfg["thresh_lo"], cfg["pseudo_weight"]

    # Select high-confidence test rows
    mask_pos  = test_df_pl["pred"] > hi
    mask_neg  = test_df_pl["pred"] < lo
    pseudo_df = test_df_pl[mask_pos | mask_neg].copy()

    # Hard labels from threshold
    pseudo_df[TARGET] = (pseudo_df["pred"] > hi).astype(int)
    pseudo_df = pseudo_df.drop(columns=["pred"])

    # Real training data
    real_feats = X.copy()
    real_feats[TARGET] = y.values
    real_feats["weight"] = 1.0

    # Pseudo-labeled data
    pseudo_feats = pseudo_df[FEATURES + [TARGET]].copy()
    pseudo_feats["weight"] = pw

    # Combine
    augmented = pd.concat([real_feats, pseudo_feats], axis=0).reset_index(drop=True)

    X_aug = augmented[FEATURES]
    y_aug = augmented[TARGET]
    w_aug = augmented["weight"]

    print(f"Config [{cfg['name']}]  thresh={lo}/{hi}  pseudo_weight={pw}")
    print(f"  Real train rows    : {len(real_feats):,}")
    print(f"  Pseudo rows added  : {len(pseudo_feats):,}  "
          f"(churn={mask_pos.sum():,} + no-churn={mask_neg.sum():,})")
    print(f"  Augmented total    : {len(augmented):,}")
    return X_aug, y_aug, w_aug

# Preview
print("── Dataset sizes per config ──")
for cfg in CONFIGS:
    make_pseudo_dataset(cfg)
    print()


In [ ]:
def run_pseudo_ensemble(cfg):
    """
    Multi-seed ensemble with pseudo-labeling.
    Trains on (real_train_fold + ALL pseudo-labels),
    validates on real_val_fold only (no leakage).
    """
    X_aug, y_aug, w_aug = make_pseudo_dataset(cfg)
    n_real   = len(X)          # boundary: first n_real rows are real data
    dtest    = xgb.DMatrix(X_test)

    test_preds_sum = np.zeros(len(X_test))
    global_oof_sum = np.zeros(len(X))
    fold_auc_log   = []

    print(f"\nEnsemble [{cfg['name']}]: {len(SEEDS)} seeds × {N_SPLITS} folds = {TOTAL_MODELS} models")
    outer_bar = tqdm(SEEDS, desc="Seeds", position=0)

    for seed in outer_bar:
        skf      = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
        seed_oof = np.zeros(len(X))

        inner_bar = tqdm(
            enumerate(skf.split(X, y)),   # split only on REAL training data
            total=N_SPLITS, desc="  Folds", position=1, leave=False,
        )

        for fold, (tr_idx, val_idx) in inner_bar:
            # Real train/val split
            X_tr_real  = X.iloc[tr_idx]
            y_tr_real  = y.iloc[tr_idx]
            X_val_real = X.iloc[val_idx]
            y_val_real = y.iloc[val_idx]

            # Pseudo rows: ALL pseudo-labeled samples (indices >= n_real in X_aug)
            X_pseudo   = X_aug.iloc[n_real:]
            y_pseudo   = y_aug.iloc[n_real:]
            w_pseudo   = w_aug.iloc[n_real:]

            # Combined train: real train fold + all pseudo-labels
            X_tr_comb  = pd.concat([X_tr_real, X_pseudo],  axis=0)
            y_tr_comb  = pd.concat([y_tr_real, y_pseudo],  axis=0)
            w_tr_comb  = np.concatenate([
                np.ones(len(X_tr_real)),
                w_pseudo.values,
            ])

            dtrain = xgb.DMatrix(X_tr_comb,  label=y_tr_comb, weight=w_tr_comb)
            dval   = xgb.DMatrix(X_val_real, label=y_val_real)

            params = {**BEST_PARAMS, "seed": seed}
            model  = xgb.train(
                params, dtrain,
                num_boost_round       = BEST_ROUNDS + 200,
                evals                 = [(dval, "val")],
                early_stopping_rounds = 100,
                verbose_eval          = False,
            )

            val_preds  = model.predict(dval,  iteration_range=(0, model.best_iteration))
            test_preds = model.predict(dtest, iteration_range=(0, model.best_iteration))

            fold_auc = roc_auc_score(y_val_real, val_preds)
            fold_auc_log.append(fold_auc)
            seed_oof[val_idx]        = val_preds
            global_oof_sum[val_idx] += val_preds
            test_preds_sum          += test_preds

            inner_bar.set_postfix({
                "fold_auc": f"{fold_auc:.5f}",
                "best_iter": model.best_iteration,
            })
            del model, dtrain, dval; gc.collect()

        seed_auc = roc_auc_score(y, seed_oof)
        outer_bar.set_postfix({"seed_oof": f"{seed_auc:.5f}"})

    global_oof = global_oof_sum / len(SEEDS)
    final_auc  = roc_auc_score(y, global_oof)
    final_preds = test_preds_sum / TOTAL_MODELS

    print(f"\n  {'='*50}")
    print(f"  [{cfg['name']}] Fold AUC : {np.mean(fold_auc_log):.5f} ± {np.std(fold_auc_log):.5f}")
    print(f"  [{cfg['name']}] OOF  AUC : {final_auc:.5f}")
    print(f"  {'='*50}")
    return final_preds, final_auc


In [ ]:
preds_A, auc_A = run_pseudo_ensemble(CONFIGS[0])   # pl_90_w05


In [ ]:
preds_B, auc_B = run_pseudo_ensemble(CONFIGS[1])   # pl_95_w05


In [ ]:
# Also blend Config A and B predictions
preds_blend_AB = (preds_A + preds_B) / 2.0

outputs = {
    "submission_pseudo_pl90_w05.csv":   preds_A,
    "submission_pseudo_pl95_w05.csv":   preds_B,
    "submission_pseudo_blend_AB.csv":   preds_blend_AB,
}

print("\n── Saving submissions ──")
for fname, preds in outputs.items():
    out = sub.copy()
    out[TARGET] = preds
    fpath = os.path.join(SUB_DIR, fname)
    out.to_csv(fpath, index=False)
    print(f"Saved → {fpath}  | range=[{preds.min():.4f}, {preds.max():.4f}] | mean={preds.mean():.4f}")

print("\n── OOF AUC Summary ──")
print(f"  Config A (thresh=0.90/0.10, w=0.5) : {auc_A:.5f}")
print(f"  Config B (thresh=0.95/0.05, w=0.5) : {auc_B:.5f}")
print(f"  Baseline nb10 (no pseudo-labels)   : ~0.914xx  (public LB reference)")


In [ ]:
import subprocess

COMPETITION = "playground-series-s6e3"

SUBMIT_THESE = [
    ("submission_pseudo_pl90_w05.csv",
     "Pseudo-label: nb10 base, thresh=0.90/0.10, pseudo_weight=0.5, 50-model ensemble"),
    ("submission_pseudo_pl95_w05.csv",
     "Pseudo-label: nb10 base, thresh=0.95/0.05, pseudo_weight=0.5, 50-model ensemble"),
    ("submission_pseudo_blend_AB.csv",
     "Pseudo-label blend: pl90+pl95 average"),
]

for fname, message in SUBMIT_THESE:
    fpath = os.path.join(SUB_DIR, fname)
    if not os.path.exists(fpath):
        print(f"⚠️  Not found: {fpath}")
        continue
    print(f"\nSubmitting: {fname}")
    r = subprocess.run(
        ["kaggle", "competitions", "submit",
         "-c", COMPETITION, "-f", fpath, "-m", message],
        capture_output=True, text=True
    )
    print(r.stdout.strip())
    if r.stderr.strip():
        print("STDERR:", r.stderr.strip())
